# Distributed Training Example

This notebook demonstrates how to use the distributed training capabilities of the Bitcoin Trading RL project.

## Setup

First, let's import the necessary modules and set up our environment.

In [ ]:
import os
import torch
import yaml
from pathlib import Path

from src.models.base_model import BaseModel
from src.training.distributed_trainer import DistributedTrainer
from src.data.binance_fetcher import BinanceFetcher
from src.features.feature_engineering import FeatureEngineer

## Configuration

Load and verify the training configuration.

In [ ]:
# Load configuration
with open('configs/config.yaml', 'r') as f:
    config = yaml.safe_load(f)

# Verify GPU availability
print(f"CUDA available: {torch.cuda.is_available()}")
if torch.cuda.is_available():
    print(f"Number of GPUs: {torch.cuda.device_count()}")
    for i in range(torch.cuda.device_count()):
        print(f"GPU {i}: {torch.cuda.get_device_name(i)}")

## Data Preparation

Prepare the training data with feature engineering.

In [ ]:
# Initialize data fetcher
fetcher = BinanceFetcher(config['data']['binance'])

# Fetch historical data
data = await fetcher.fetch_historical_data()

# Initialize feature engineer
engineer = FeatureEngineer(
    input_file='data/raw/BTCUSDT_data.parquet',
    output_dir='data/processed',
    config=config
)

# Generate features
await engineer.generate_features()

## Dataset Creation

Create PyTorch datasets for training and validation.

In [ ]:
class TradingDataset(torch.utils.data.Dataset):
    def __init__(self, features, targets):
        self.features = torch.FloatTensor(features)
        self.targets = torch.FloatTensor(targets)
    
    def __len__(self):
        return len(self.features)
    
    def __getitem__(self, idx):
        return self.features[idx], self.targets[idx]

# Load processed features
processed_data = pd.read_parquet('data/processed/features_latest.parquet')

# Prepare features and targets
features = processed_data.drop(['open_time', 'target'], axis=1).values
targets = processed_data['target'].values

# Split data
split_idx = int(len(features) * 0.8)
train_dataset = TradingDataset(features[:split_idx], targets[:split_idx])
val_dataset = TradingDataset(features[split_idx:], targets[split_idx:])

## Single GPU Training

First, let's try training on a single GPU.

In [ ]:
# Initialize model and trainer
model = BaseModel()
trainer = DistributedTrainer(
    model=model,
    config=config['training'],
    world_size=1
)

# Train model
history = trainer.train(
    train_dataset=train_dataset,
    val_dataset=val_dataset,
    num_epochs=config['training']['num_epochs'],
    batch_size=config['training']['batch_size'],
    learning_rate=config['training']['learning_rate']
)

## Multi-GPU Training

Now, let's scale up to multiple GPUs.

In [ ]:
import torch.multiprocessing as mp

def train_distributed(rank, world_size, config):
    # Initialize model and trainer
    model = BaseModel()
    trainer = DistributedTrainer(
        model=model,
        config=config['training'],
        world_size=world_size,
        rank=rank
    )
    
    # Train model
    history = trainer.train(
        train_dataset=train_dataset,
        val_dataset=val_dataset,
        num_epochs=config['training']['num_epochs'],
        batch_size=config['training']['batch_size'],
        learning_rate=config['training']['learning_rate']
    )
    
    return history

# Start multi-GPU training
if torch.cuda.is_available() and torch.cuda.device_count() > 1:
    world_size = torch.cuda.device_count()
    mp.spawn(
        train_distributed,
        args=(world_size, config),
        nprocs=world_size,
        join=True
    )

## Performance Analysis

Compare training performance between single and multi-GPU setups.

In [ ]:
import matplotlib.pyplot as plt

def plot_training_comparison(single_gpu_history, multi_gpu_history):
    fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(15, 5))
    
    # Plot training loss
    ax1.plot(single_gpu_history['train_loss'], label='Single GPU')
    ax1.plot(multi_gpu_history['train_loss'], label='Multi GPU')
    ax1.set_title('Training Loss')
    ax1.set_xlabel('Epoch')
    ax1.set_ylabel('Loss')
    ax1.legend()
    
    # Plot validation loss
    ax2.plot(single_gpu_history['val_loss'], label='Single GPU')
    ax2.plot(multi_gpu_history['val_loss'], label='Multi GPU')
    ax2.set_title('Validation Loss')
    ax2.set_xlabel('Epoch')
    ax2.set_ylabel('Loss')
    ax2.legend()
    
    plt.tight_layout()
    plt.show()

# Plot comparison
plot_training_comparison(single_gpu_history, multi_gpu_history)

## Memory Usage Analysis

Monitor GPU memory usage during training.

In [ ]:
def print_gpu_memory_stats():
    if torch.cuda.is_available():
        for i in range(torch.cuda.device_count()):
            print(f"GPU {i} Memory Usage:")
            print(f"Allocated: {torch.cuda.memory_allocated(i) / 1e9:.2f} GB")
            print(f"Cached: {torch.cuda.memory_reserved(i) / 1e9:.2f} GB")

print_gpu_memory_stats()

## Save Results

Save the trained model and training history.

In [ ]:
# Save model
save_path = Path('results/models')
save_path.mkdir(parents=True, exist_ok=True)

torch.save({
    'model_state_dict': model.state_dict(),
    'training_history': history,
    'config': config
}, save_path / 'distributed_training_result.pt')